# Heritage Short: ACE-Step 1.5 score candidates (Kaggle T4 / P100 GPU)

Reads `jobs.json` (made by `python -m pipeline.music export`; upload it as a dataset or
drop it in `/kaggle/working`), generates each candidate as `<id>__<key>.flac`, and logs every
call to `runs_kaggle.jsonl`. Bring them back with `python -m pipeline.music import <folder>`.

Uses the current ACE-Step 1.5 Python API (`AceStepHandler` + `LLMHandler` with `backend="pt"`
for Kaggle T4/P100 compatibility without `vllm`, plus `offload_to_cpu=True` for <20 GB VRAM).

**Kaggle runs Python 3.13 but ACE-Step 1.5 needs 3.11-3.12** (FAILURES.md F7), so the notebook builds a Python 3.12 venv with `uv` and runs generation there. The repo is cloned to `/tmp` so `kaggle kernels output` stays small.


In [ ]:
%%bash
set -x
pip -q install uv
uv python install 3.12
uv venv /tmp/venv --python 3.12
git clone -q https://github.com/ace-step/ACE-Step-1.5.git /tmp/ace15 || true
cd /tmp/ace15
# flash-attn cannot build on Kaggle (no nvcc); it is optional, so drop it from the requirements.
sed -i '/flash[-_]attn/d' requirements.txt pyproject.toml 2>/dev/null || true
uv pip install --python /tmp/venv/bin/python -e . 2>&1 | tail -15 || true
/tmp/venv/bin/python -c "import acestep, torch; print('acestep OK, torch', torch.__version__, 'cuda', torch.cuda.is_available())"


In [ ]:
GEN = r"""
import glob, json, os, shutil, time, datetime, pathlib, torch
cands = glob.glob('/kaggle/input/**/jobs.json', recursive=True) + glob.glob('/kaggle/working/jobs.json')
assert cands, 'jobs.json not found: upload output of `python -m pipeline.music export`'
jobs = json.load(open(cands[0], encoding='utf-8'))['jobs']
OUT = pathlib.Path('/kaggle/working/music'); OUT.mkdir(parents=True, exist_ok=True)
print(len(jobs), 'candidates from', cands[0])
from acestep.handler import AceStepHandler
from acestep.llm_inference import LLMHandler
from acestep.inference import GenerationParams, GenerationConfig, generate_music

ROOT = '/tmp/ace15'
CKPT = os.path.join(ROOT, 'checkpoints')
dit = AceStepHandler()
dit.initialize_service(
    project_root=ROOT,
    config_path='acestep-v15-turbo',
    device='cuda',
    offload_to_cpu=True,
    quantization='int8_weight_only',
)
llm = LLMHandler()
try:
    llm.initialize(
        checkpoint_dir=CKPT,
        lm_model_path='acestep-5Hz-lm-0.6B',
        backend='pt',
        device='cuda',
        offload_to_cpu=True,
    )
except Exception as e:
    print('LLMHandler init failed, continuing without LM (bpm/key set explicitly):', e)
    llm = None
log_path = pathlib.Path('/kaggle/working/runs_kaggle.jsonl')
with open(log_path, 'a', encoding='utf-8') as log:
    for j in jobs:
        out_name = pathlib.Path(j['output']).with_suffix('.flac').name
        out = OUT / out_name
        caption = j['caption'] + (', ' + ', '.join(j.get('tags', [])) if j.get('tags') else '')
        caption = caption[:512]
        t = time.time()
        gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'
        rec = {
            'ts': datetime.datetime.utcnow().isoformat(timespec='seconds') + 'Z',
            'run_id': 'kaggle-music',
            'stage': 'music',
            'item': j['id'],
            'provider': 'kaggle',
            'model': 'ACE-Step-1.5-turbo',
            'seed': j['seed'],
            'cache_key': j['key'],
            'params': {'caption': caption, 'bpm': j.get('bpm', 66), 'key': j.get('key', 'D minor'), 'duration_s': j['duration_s']},
            'cost_usd': 0.0,
            'gpu': gpu_name,
        }
        try:
            params = GenerationParams(
                caption=caption,
                lyrics=j.get('lyrics', '[Instrumental]'),
                instrumental=True,
                bpm=j.get('bpm') or 66,
                keyscale=j.get('key') or 'D minor',
                duration=j['duration_s'],
                seed=j['seed'],
                shift=3.0,
            )
            config = GenerationConfig(batch_size=1, use_random_seed=False, audio_format='flac')
            res = generate_music(dit, llm, params, config, save_dir=str(OUT))
            src_path = pathlib.Path(res.audios[0]['path'])
            shutil.move(str(src_path), str(out))
            rec['status'] = 'ok'
            rec['output'] = out.name
        except Exception as e:
            rec['status'] = 'fatal_error'
            rec['error'] = str(e)[:500]
        dt = round(time.time() - t, 1)
        rec['latency_s'] = rec['gpu_s'] = dt
        log.write(json.dumps(rec) + '\n')
        log.flush()
        print(j['id'], rec['status'], dt, 's')
"""


In [ ]:
import pathlib, subprocess
pathlib.Path('/tmp/gen.py').write_text(GEN)
import os
env = {k: v for k, v in os.environ.items() if k not in ('MPLBACKEND',)}
env['MPLBACKEND'] = 'Agg'  # the notebook's inline backend does not exist in the venv (FAILURES.md F7)
r = subprocess.run(['/tmp/venv/bin/python', '/tmp/gen.py'], env=env)
print('generation exit code', r.returncode)
assert r.returncode == 0
